# Taller 5 – BI · Airbnb Ciudad de México
## Segmentación K-Means y Visualizaciones Analíticas


## 1. Configuración e Importación de Librerías


In [55]:
import os
import sqlite3
import sys
import textwrap
import warnings
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings("ignore")

# ──────────────────────────────────────────
# CONFIGURACIÓN GLOBAL
# ──────────────────────────────────────────
DB_PATH = "../data/airbnb.db"
OUTPUT_DIR = "v1_visualizations"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

PALETTE = "viridis"
COLOR_MAIN = "#1a6b9a"
COLOR_ACC = "#e87d37"
PLT_STYLE = "seaborn-v0_8-whitegrid"
plt.style.use(PLT_STYLE)

print("Librerías cargadas correctamente.")
print(f"Directorio de salida: {OUTPUT_DIR}/")


Librerías cargadas correctamente.
Directorio de salida: v1_visualizations/


## 2. Utilidades y Funciones Auxiliares


In [56]:
SEP = "=" * 72


def sep(title=""):
    if title:
        pad = (72 - len(title) - 2) // 2
        print(f"\n{'=' * pad} {title} {'=' * pad}\n")
    else:
        print(f"\n{SEP}\n")


def save_fig(name: str):
    path = os.path.join(OUTPUT_DIR, f"{name}.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  [FIG guardada] → {path}")


def normalizar_available(series: pd.Series) -> pd.Series:
    """Acepta 0/1 enteros o 't'/'f' strings → retorna 0/1 int."""
    if pd.api.types.is_numeric_dtype(series):
        return series.fillna(0).astype(int)
    return series.map({"t": 1, "f": 0, True: 1, False: 0}).fillna(0).astype(int)


def limpiar_listings(df: pd.DataFrame) -> pd.DataFrame:
    """Prepara Listings para análisis: convierte tipos, imputa nulos básicos."""
    df = df.copy()
    # Precio
    if "price" in df.columns:
        df["price"] = pd.to_numeric(df["price"], errors="coerce")
        df = df[df["price"].notna() & (df["price"] > 0) & (df["price"] < 50_000)]
    # Columnas numéricas clave
    num_cols = [
        "accommodates",
        "bedrooms",
        "beds",
        "number_of_reviews",
        "reviews_per_month",
        "review_scores_rating",
        "minimum_nights",
        "maximum_nights",
        "calculated_host_listings_count",
        "availability_365",
    ]
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    # Booleanos
    for c in ["host_is_superhost", "instant_bookable"]:
        if c in df.columns:
            df[c] = df[c].map({"t": 1, "f": 0, True: 1, False: 0})
    return df


print("Funciones auxiliares definidas.")


Funciones auxiliares definidas.


## 3. Carga de Datos desde el ETL

Los datos provienen del proceso ETL almacenado en SQLite (`airbnb.db`).

Se cargan las tablas `Listings`, `Calendar` y `Reviews`.


In [57]:
def cargar_datos(db_path: str) -> dict[str, pd.DataFrame]:
    if not Path(db_path).exists():
        sys.exit(f"[ERROR] Base de datos no encontrada en: {db_path}")
    conn = sqlite3.connect(db_path)
    dfs = {}
    tablas = ["Listings", "Calendar", "Reviews"]
    for t in tablas:
        try:
            dfs[t] = pd.read_sql(f'SELECT * FROM "{t}"', conn)
            print(
                f"  [{t}] cargada: {len(dfs[t]):,} filas × {dfs[t].shape[1]} columnas"
            )
        except Exception as e:
            print(f"  [WARN] No se pudo cargar {t}: {e}")
            dfs[t] = pd.DataFrame()
    conn.close()
    return dfs


sep("TALLER 5 – BI · AIRBNB CDMX")
print(f"  Base de datos : {DB_PATH}")
print(f"  Salida figuras: {OUTPUT_DIR}/")
print()

print("Cargando datos desde SQLite...")
dfs = cargar_datos(DB_PATH)



===================== TALLER 5 – BI · AIRBNB CDMX =====================

  Base de datos : ../data/airbnb.db
  Salida figuras: v1_visualizations/

Cargando datos desde SQLite...
  [Listings] cargada: 23,650 filas × 80 columnas
  [Calendar] cargada: 165,345 filas × 9 columnas
  [Reviews] cargada: 27,892 filas × 10 columnas


## 4. Punto 3 – Visualizaciones Analíticas y Métricas

Se generan visualizaciones sobre precios, distribución geográfica,
disponibilidad y tendencias temporales de reseñas.


In [58]:

sep("PUNTO 3 · VISUALIZACIONES ANALÍTICAS Y MÉTRICAS")

listings = limpiar_listings(dfs["Listings"])
calendar = dfs["Calendar"].copy()
reviews  = dfs["Reviews"].copy()

if "available" in calendar.columns:
    calendar["available"] = normalizar_available(calendar["available"])

print(f"Listings limpios : {len(listings):,} registros")
print(f"Calendar         : {len(calendar):,} registros")
print(f"Reviews          : {len(reviews):,} registros")



=========== PUNTO 3 · VISUALIZACIONES ANALÍTICAS Y MÉTRICAS ===========

Listings limpios : 21,018 registros
Calendar         : 165,345 registros
Reviews          : 27,892 registros


In [59]:
### 4.1 VIZ-1 · KPI Dashboard General

# %%
print("\n[VIZ-1] KPI Dashboard general")

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.patch.set_facecolor("#f0f4f8")

kpi_colors = [
    COLOR_MAIN, "#2e7d32", "#6a1b9a", "#ad1457",
    "#00695c", "#e65100", "#37474f", "#1a237e",
]


def kpi_box(ax, value, label, color=COLOR_MAIN):
    ax.set_facecolor(color)
    ax.text(
        0.5, 0.58, str(value),
        ha="center", va="center", fontsize=22, fontweight="bold",
        color="white", transform=ax.transAxes,
    )
    ax.text(
        0.5, 0.22, label,
        ha="center", va="center", fontsize=10,
        color="white", transform=ax.transAxes,
    )
    ax.axis("off")


kpi_data = [
    (f"{len(listings):,}", "Total Listings"),
    (
        f"${listings['price'].median():.0f}" if "price" in listings.columns else "N/D",
        "Precio Mediano\n(MXN)",
    ),
    (
        f"${listings['price'].mean():.0f}" if "price" in listings.columns else "N/D",
        "Precio Promedio\n(MXN)",
    ),
    (
        f"{listings['room_type'].nunique() if 'room_type' in listings.columns else 'N/D'}",
        "Tipos de Cuarto",
    ),
    (
        f"{listings['neighbourhood_cleansed'].nunique() if 'neighbourhood_cleansed' in listings.columns else 'N/D'}",
        "Colonias",
    ),
    (
        f"{listings['host_id'].nunique() if 'host_id' in listings.columns else 'N/D'}",
        "Anfitriones",
    ),
    (f"{len(reviews):,}", "Total Reseñas"),
    (
        f"{calendar['available'].mean() * 100:.1f}%"
        if not calendar.empty and "available" in calendar.columns
        else "N/D",
        "Disponibilidad\nPromedio",
    ),
]

for ax, (val, lbl), col in zip(axes.flat, kpi_data, kpi_colors):
    kpi_box(ax, val, lbl, col)

fig.suptitle(
    "KPIs Generales – Airbnb Ciudad de México",
    fontsize=15, fontweight="bold", y=1.01,
)
plt.tight_layout(pad=0.5)
save_fig("03a_kpi_dashboard")

# %% [markdown]
### 4.2 VIZ-2 · Distribución de Precios por Tipo de Alojamiento

# %%
print("\n[VIZ-2] Distribución de precios por tipo de alojamiento (PN-3)")

if "room_type" in listings.columns and "price" in listings.columns:
    p99 = listings["price"].quantile(0.99)
    df_viz = listings[listings["price"] <= p99].copy()

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Boxplot
    order = (
        df_viz.groupby("room_type")["price"]
        .median()
        .sort_values(ascending=False)
        .index
    )
    sns.boxplot(
        data=df_viz, x="room_type", y="price",
        order=order, palette="Blues_d", ax=axes[0],
    )
    axes[0].set_title(
        "Distribución de Precio por Tipo de Cuarto\n(sin outliers >p99)",
        fontweight="bold",
    )
    axes[0].set_xlabel("Tipo de alojamiento")
    axes[0].set_ylabel("Precio (MXN/noche)")
    axes[0].yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:,.0f}")
    )
    axes[0].tick_params(axis="x", rotation=15)

    # Precio mediano por tipo
    agg = (
        df_viz.groupby("room_type")["price"]
        .agg(conteo="count", mediana="median", promedio="mean")
        .sort_values("mediana", ascending=True)
    )
    agg["mediana"].plot(kind="barh", color=COLOR_MAIN, ax=axes[1])
    axes[1].set_title("Precio Mediano por Tipo de Cuarto", fontweight="bold")
    axes[1].set_xlabel("Precio Mediano (MXN)")
    axes[1].set_ylabel("")
    axes[1].xaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:,.0f}")
    )
    for i, (idx, row) in enumerate(agg.iterrows()):
        axes[1].text(
            row["mediana"] + 1, i, f"  n={row['conteo']:,}", va="center", fontsize=9
        )

    plt.tight_layout()
    save_fig("03b_precio_por_room_type")
else:
    print("  [SKIP] Columnas room_type o price no disponibles.")

# %% [markdown]
### 4.3 VIZ-3 · Top Colonias por Precio Mediano

# %%
print("\n[VIZ-3] Top colonias por precio mediano (PN-1)")

if "neighbourhood_cleansed" in listings.columns and "price" in listings.columns:
    top_n = 20
    colonia_stats = (
        listings.groupby("neighbourhood_cleansed")["price"]
        .agg(mediana="median", conteo="count", promedio="mean")
        .query("conteo >= 10")
        .sort_values("mediana", ascending=False)
        .head(top_n)
    )

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # Bar chart precio mediano
    colores = plt.cm.Blues_r(np.linspace(0.2, 0.8, len(colonia_stats)))
    colonia_stats["mediana"].sort_values().plot(
        kind="barh", color=colores[::-1], ax=axes[0]
    )
    axes[0].set_title(
        f"Top {top_n} Alcaldías – Precio Mediano (MXN)\n(mín. 10 alojamientos)",
        fontweight="bold",
    )
    axes[0].set_xlabel("Precio Mediano (MXN)")
    axes[0].xaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:,.0f}")
    )

    # Scatter: volumen vs precio mediano
    axes[1].scatter(
        colonia_stats["conteo"], colonia_stats["mediana"],
        color=COLOR_MAIN, alpha=0.7, s=80, edgecolors="white",
    )
    for row in colonia_stats.itertuples():
        axes[1].annotate(
            row.Index[:15], (row.conteo, row.mediana),
            fontsize=7, alpha=0.75, xytext=(4, 2), textcoords="offset points",
        )
    axes[1].set_xlabel("Número de alojamientos")
    axes[1].set_ylabel("Precio Mediano (MXN)")
    axes[1].set_title("Volumen vs. Precio Mediano por Alcaldía", fontweight="bold")
    axes[1].yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:,.0f}")
    )

    plt.tight_layout()
    save_fig("03c_top_colonias_precio")
else:
    print("  [SKIP] Columnas neighbourhood_cleansed o price no disponibles.")

# %% [markdown]
### 4.4 VIZ-5 · Evolución Temporal de Reseñas

# %%
print("\n[VIZ-5] Evolución temporal de reseñas (PN-4)")

if "año" in reviews.columns and "mes" in reviews.columns:
    reviews["año"] = pd.to_numeric(reviews["año"], errors="coerce")
    reviews["mes"] = pd.to_numeric(reviews["mes"], errors="coerce")
    rev_time = (
        reviews.dropna(subset=["año", "mes"])
        .groupby(["año", "mes"])
        .size()
        .reset_index(name="conteo")
    )
    rev_time["periodo"] = pd.to_datetime(
        rev_time["año"].astype(int).astype(str)
        + "-"
        + rev_time["mes"].astype(int).astype(str).str.zfill(2)
    )
    rev_time = rev_time.sort_values("periodo")
    rev_time = rev_time[rev_time["año"] >= 2015]

    fig, axes = plt.subplots(2, 1, figsize=(14, 9))

    # Serie de tiempo
    axes[0].fill_between(
        rev_time["periodo"], rev_time["conteo"], alpha=0.3, color=COLOR_MAIN
    )
    axes[0].plot(
        rev_time["periodo"], rev_time["conteo"], color=COLOR_MAIN, linewidth=1.5
    )
    axes[0].set_title(
        "Evolución Mensual del Volumen de Reseñas – CDMX", fontweight="bold"
    )
    axes[0].set_ylabel("Número de reseñas")
    axes[0].yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"{x:,.0f}")
    )
    axes[0].set_xlabel("")

    # Heatmap estacionalidad (año × mes)
    heat = rev_time.pivot_table(
        index="año", columns="mes", values="conteo", aggfunc="sum"
    )
    heat.columns = [
        "Ene", "Feb", "Mar", "Abr", "May", "Jun",
        "Jul", "Ago", "Sep", "Oct", "Nov", "Dic",
    ][: len(heat.columns)]
    sns.heatmap(
        heat, cmap="YlOrRd", annot=True, fmt=".0f",
        linewidths=0.5, ax=axes[1], annot_kws={"size": 8},
    )
    axes[1].set_title(
        "Heatmap de Estacionalidad – Reseñas por Año y Mes", fontweight="bold"
    )
    axes[1].set_xlabel("Mes")
    axes[1].set_ylabel("Año")

    plt.tight_layout()
    save_fig("03e_evolucion_reseñas_temporal")

    peak = rev_time.loc[rev_time["conteo"].idxmax()]
    print(
        f"\n  Pico máximo de reseñas: {peak['conteo']:,} en "
        f"{peak['periodo'].strftime('%B %Y')}"
    )
    print(f"  Total reseñas analizadas: {rev_time['conteo'].sum():,}")
else:
    print("  [SKIP] Columnas año/mes no disponibles en Reviews.")

# %% [markdown]
### 4.5 VIZ-6 · Disponibilidad Mensual en Calendar

# %%
print("\n[VIZ-6] Disponibilidad mensual en Calendar (PN-4)")

if "mes" in calendar.columns and "available" in calendar.columns and not calendar.empty:
    calendar["mes"] = pd.to_numeric(calendar["mes"], errors="coerce")
    disp_mes = (
        calendar.dropna(subset=["mes"])
        .groupby("mes")["available"]
        .mean()
        .reset_index()
    )
    disp_mes["mes_nombre"] = disp_mes["mes"].map({
        1: "Ene", 2: "Feb", 3: "Mar",  4: "Abr",
        5: "May", 6: "Jun", 7: "Jul",  8: "Ago",
        9: "Sep", 10: "Oct", 11: "Nov", 12: "Dic",
    })
    disp_mes["tasa_pct"] = disp_mes["available"] * 100

    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.bar(
        disp_mes["mes_nombre"], disp_mes["tasa_pct"],
        color=COLOR_MAIN, alpha=0.8, edgecolor="white",
    )
    ax.set_title(
        "Tasa de Disponibilidad Promedio por Mes – Calendar", fontweight="bold"
    )
    ax.set_ylabel("% días disponibles")
    ax.set_ylim(0, 100)
    for bar, val in zip(bars, disp_mes["tasa_pct"]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1,
            f"{val:.1f}%",
            ha="center", va="bottom", fontsize=9,
        )
    ax.axhline(
        disp_mes["tasa_pct"].mean(),
        color=COLOR_ACC, linestyle="--",
        label=f"Promedio general: {disp_mes['tasa_pct'].mean():.1f}%",
    )
    ax.legend()
    plt.tight_layout()
    save_fig("03f_disponibilidad_mensual")
else:
    print("  [SKIP] Datos de Calendar insuficientes.")

# %% [markdown]
### 4.6 VIZ-7 · Distribución de Segmentos de Precio (price_rango)

# %%
print("\n[VIZ-7] Distribución de segmentos de precio (price_rango)")

if "price_rango" in listings.columns and "room_type" in listings.columns:
    cross = pd.crosstab(listings["room_type"], listings["price_rango"])
    cross_pct = cross.div(cross.sum(axis=1), axis=0) * 100

    fig, ax = plt.subplots(figsize=(11, 5))
    cross_pct.plot(kind="bar", stacked=True, colormap="Blues", ax=ax)
    ax.set_title(
        "Composición de Segmento de Precio por Tipo de Alojamiento\n"
        "(% dentro de cada room_type)",
        fontweight="bold",
    )
    ax.set_ylabel("% de listings")
    ax.set_xlabel("Tipo de cuarto")
    ax.legend(title="Rango de Precio", bbox_to_anchor=(1.01, 1), loc="upper left")
    ax.tick_params(axis="x", rotation=15)
    plt.tight_layout()
    save_fig("03g_segmentos_precio_room_type")
else:
    print("  [SKIP] Columnas price_rango o room_type no disponibles.")



[VIZ-1] KPI Dashboard general
  [FIG guardada] → v1_visualizations\03a_kpi_dashboard.png

[VIZ-2] Distribución de precios por tipo de alojamiento (PN-3)
  [FIG guardada] → v1_visualizations\03b_precio_por_room_type.png

[VIZ-3] Top colonias por precio mediano (PN-1)
  [FIG guardada] → v1_visualizations\03c_top_colonias_precio.png

[VIZ-5] Evolución temporal de reseñas (PN-4)
  [FIG guardada] → v1_visualizations\03e_evolucion_reseñas_temporal.png

  Pico máximo de reseñas: 712 en August 2025
  Total reseñas analizadas: 27,863

[VIZ-6] Disponibilidad mensual en Calendar (PN-4)
  [FIG guardada] → v1_visualizations\03f_disponibilidad_mensual.png

[VIZ-7] Distribución de segmentos de precio (price_rango)
  [FIG guardada] → v1_visualizations\03g_segmentos_precio_room_type.png


## 5. Punto 4 – Modelo de Machine Learning: Segmentación K-Means

 **Objetivo:** Segmentar los alojamientos de Airbnb CDMX en grupos homogéneos
 según características de precio, capacidad, calidad percibida y comportamiento
 de demanda, sin supervisión previa.

 **Justificación del enfoque:**
 - No existe una variable objetivo etiquetada → aprendizaje no supervisado.
 - K-Means agrupa por similitud en espacio métrico multivariado, revelando
   segmentos que combinan precio, capacidad y demanda.
 - Complementa PN-3: la categoría `room_type` es autodeclarada por el anfitrión;
   K-Means detecta si el mercado real sigue esa lógica.


==== PUNTO 4 · MODELO DE ML – SEGMENTACIÓN K-MEANS DE ALOJAMIENTOS ====

Segmentar los alojamientos de Airbnb CDMX en grupos homogéneos según sus características de precio, capacidad, calidad percibida y comportamiento de demanda, sin supervisión previa. El clustering responde a PN-3: ¿existen segmentos diferenciados con dinámicas propias más allá de la categoría room_type declarada?


In [60]:
sep("PUNTO 4 · MODELO DE ML – SEGMENTACIÓN K-MEANS DE ALOJAMIENTOS")



### 5.1 Preparación de Datos para el Modelo

# %%
print("\n--- PREPARACIÓN DE DATOS ---")

listings_ml = limpiar_listings(dfs["Listings"])

FEATURES = [
    c for c in [
        "price", "accommodates", "bedrooms", "beds",
        "number_of_reviews", "reviews_per_month",
        "review_scores_rating", "minimum_nights",
        "availability_365", "calculated_host_listings_count",
    ]
    if c in listings_ml.columns
]

COLS_PERFIL = [
    c for c in [
        "room_type", "neighbourhood_cleansed",
        "property_type", "host_is_superhost",
    ]
    if c in listings_ml.columns
]

df_ml = listings_ml[FEATURES + COLS_PERFIL].copy()
antes = len(df_ml)
df_ml = df_ml.dropna(subset=FEATURES)
despues = len(df_ml)

print(f"  Registros antes de limpieza : {antes:,}")
print(f"  Registros para clustering   : {despues:,} ({antes - despues} eliminados por nulos)")
print(f"  Variables de entrada        : {FEATURES}")
print(f"  Variables para perfilado    : {COLS_PERFIL}")

X = df_ml[FEATURES].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\nEscalado completado (StandardScaler).")



==== PUNTO 4 · MODELO DE ML – SEGMENTACIÓN K-MEANS DE ALOJAMIENTOS ====


--- PREPARACIÓN DE DATOS ---
  Registros antes de limpieza : 21,018
  Registros para clustering   : 21,018 (0 eliminados por nulos)
  Variables de entrada        : ['price', 'accommodates', 'bedrooms', 'beds', 'number_of_reviews', 'reviews_per_month', 'review_scores_rating', 'minimum_nights', 'availability_365', 'calculated_host_listings_count']
  Variables para perfilado    : ['room_type', 'neighbourhood_cleansed', 'property_type', 'host_is_superhost']

Escalado completado (StandardScaler).


### 5.2 Determinación del Número Óptimo de Clusters
Se evalúan k=2 a k=10 con tres métricas complementarias:
- **Método del Codo (Elbow):** minimizar la inercia intra-cluster (WCSS).
- **Coeficiente de Silueta:** mide qué tan bien separados están los clusters.
- **Davies-Bouldin Score:** menor valor = mejor separación.


In [61]:
print("\n--- DETERMINACIÓN DE K ÓPTIMO (Elbow + Silhouette + Davies-Bouldin) ---")

K_RANGE = range(2, 11)
inercias    = []
silhouettes = []
db_scores   = []
ch_scores   = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, init="k-means++", n_init=10, max_iter=300, random_state=42)
    labels_k = km.fit_predict(X_scaled)
    inercias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels_k, sample_size=3000, random_state=42))
    db_scores.append(davies_bouldin_score(X_scaled, labels_k))
    ch_scores.append(calinski_harabasz_score(X_scaled, labels_k))
    print(
        f"  k={k}  Inercia={km.inertia_:>12,.1f}  "
        f"Silhouette={silhouettes[-1]:.4f}  "
        f"Davies-Bouldin={db_scores[-1]:.4f}"
    )

# Selección automática por máximo Silhouette
k_opt_idx = int(np.argmax(silhouettes))
k_opt     = list(K_RANGE)[k_opt_idx]
print(
    f"\n  → k óptimo por Silhouette máximo: k={k_opt} "
    f"(score={silhouettes[k_opt_idx]:.4f})"
)



--- DETERMINACIÓN DE K ÓPTIMO (Elbow + Silhouette + Davies-Bouldin) ---
  k=2  Inercia=   178,834.0  Silhouette=0.4108  Davies-Bouldin=1.4385
  k=3  Inercia=   161,844.7  Silhouette=0.2414  Davies-Bouldin=1.5691
  k=4  Inercia=   145,549.0  Silhouette=0.2487  Davies-Bouldin=1.2393
  k=5  Inercia=   130,295.0  Silhouette=0.2833  Davies-Bouldin=1.1824
  k=6  Inercia=   116,014.3  Silhouette=0.2225  Davies-Bouldin=1.1980
  k=7  Inercia=   104,984.9  Silhouette=0.2156  Davies-Bouldin=1.1374
  k=8  Inercia=    94,392.4  Silhouette=0.2310  Davies-Bouldin=1.0665
  k=9  Inercia=    89,385.4  Silhouette=0.2254  Davies-Bouldin=1.1358
  k=10  Inercia=    85,080.8  Silhouette=0.2354  Davies-Bouldin=1.0387

  → k óptimo por Silhouette máximo: k=2 (score=0.4108)


### 5.3 Entrenamiento del Modelo Final


In [62]:

print(f"\n--- ENTRENAMIENTO FINAL CON k={k_opt} ---")

kmeans_final = KMeans(
    n_clusters=k_opt, init="k-means++", n_init=20, max_iter=500, random_state=42
)
df_ml["cluster"] = kmeans_final.fit_predict(X_scaled)

sil_final = silhouette_score(X_scaled, df_ml["cluster"], sample_size=5000, random_state=42)
db_final  = davies_bouldin_score(X_scaled, df_ml["cluster"])
ch_final  = calinski_harabasz_score(X_scaled, df_ml["cluster"])

print(f"  Silhouette Score     : {sil_final:.4f}  (más alto = clusters más compactos/separados)")
print(f"  Davies-Bouldin Score : {db_final:.4f}  (más bajo = mejor separación)")
print(f"  Calinski-Harabasz    : {ch_final:.2f}  (más alto = clusters más densos)")



--- ENTRENAMIENTO FINAL CON k=2 ---
  Silhouette Score     : 0.4182  (más alto = clusters más compactos/separados)
  Davies-Bouldin Score : 1.4388  (más bajo = mejor separación)
  Calinski-Harabasz    : 3683.70  (más alto = clusters más densos)


In [63]:
### 5.4 Perfilado e Interpretación de los Clusters

# %%
print("\n--- PERFIL DE CLUSTERS ---")

perfil_num = df_ml.groupby("cluster")[FEATURES].agg(["mean", "median"]).round(2)
print(perfil_num.T.to_string())

# Distribución de room_type por cluster
if "room_type" in df_ml.columns:
    print("\n  Distribución de room_type por cluster (%):")
    cross_rt = (
        pd.crosstab(df_ml["cluster"], df_ml["room_type"], normalize="index") * 100
    )
    print(cross_rt.round(1).to_string())

if "neighbourhood_cleansed" in df_ml.columns:
    print("\n  Alcaldía dominante por cluster:")
    for c in sorted(df_ml["cluster"].unique()):
        top_nb = (
            df_ml[df_ml["cluster"] == c]["neighbourhood_cleansed"]
            .value_counts()
            .index[0]
        )
        n   = (df_ml["cluster"] == c).sum()
        pct = n / len(df_ml) * 100
        print(
            f"  Cluster {c}: {n:>5,} listings ({pct:.1f}%)  |  "
            f"Alcaldía predominante: {top_nb}"
        )



--- PERFIL DE CLUSTERS ---
cluster                                      0        1
price                          mean    1111.47  3653.65
                               median   950.00  2458.00
accommodates                   mean       2.83     8.07
                               median     2.00     7.00
bedrooms                       mean       1.28     3.41
                               median     1.00     3.00
beds                           mean       1.59     4.97
                               median     1.00     4.00
number_of_reviews              mean      64.75    65.81
                               median    30.00    38.00
reviews_per_month              mean       1.99     1.91
                               median     1.38     1.62
review_scores_rating           mean       4.75     4.78
                               median     4.84     4.85
minimum_nights                 mean       2.96     3.82
                               median     1.00     2.00
availability_365    

In [64]:

# Tabla resumen
print("\n  Resumen estadístico por cluster:")
resumen_cols = [
    c for c in [
        "price", "accommodates", "bedrooms",
        "review_scores_rating", "availability_365", "number_of_reviews",
    ]
    if c in FEATURES
]
resumen = df_ml.groupby("cluster")[resumen_cols].mean().round(1)
resumen.insert(0, "n_listings", df_ml.groupby("cluster").size())
resumen["% total"] = (resumen["n_listings"] / len(df_ml) * 100).round(1)
print(resumen.to_string())



  Resumen estadístico por cluster:
         n_listings   price  accommodates  bedrooms  review_scores_rating  availability_365  number_of_reviews  % total
cluster                                                                                                                
0             18667  1111.5           2.8       1.3                   4.8             252.7               64.7     88.8
1              2351  3653.6           8.1       3.4                   4.8             237.9               65.8     11.2


## 6. Visualizaciones del Modelo K-Means

Panel completo con: Elbow, Silhouette, Davies-Bouldin, proyección PCA,

boxplot de precios y perfil de centroides normalizados.


In [65]:
CLUSTER_PALETTE = sns.color_palette("tab10", n_colors=k_opt)

# PCA 2D para proyección
pca     = PCA(n_components=2, random_state=42)
X_pca   = pca.fit_transform(X_scaled)
var_exp = pca.explained_variance_ratio_
print(
    f"\n  PCA – varianza explicada: PC1={var_exp[0]:.3f}, PC2={var_exp[1]:.3f} "
    f"(total={sum(var_exp):.3f})"
)

fig = plt.figure(figsize=(20, 14))
gs  = fig.add_gridspec(3, 3, hspace=0.40, wspace=0.35)

# 4a: Elbow (inercia)
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(list(K_RANGE), inercias, "o-", color=COLOR_MAIN, linewidth=2, markersize=6)
ax1.axvline(k_opt, color=COLOR_ACC, linestyle="--", label=f"k óptimo = {k_opt}", linewidth=1.5)
ax1.set_title("Método Elbow – Inercia vs k", fontweight="bold")
ax1.set_xlabel("Número de clusters (k)")
ax1.set_ylabel("Inercia (WCSS)")
ax1.legend(fontsize=8)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x / 1e6:.1f}M"))

# 4b: Silhouette vs k
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(list(K_RANGE), silhouettes, "s-", color=COLOR_ACC, linewidth=2, markersize=6)
ax2.axvline(k_opt, color=COLOR_MAIN, linestyle="--", label=f"k óptimo = {k_opt}", linewidth=1.5)
ax2.set_title("Silhouette Score vs k", fontweight="bold")
ax2.set_xlabel("Número de clusters (k)")
ax2.set_ylabel("Silhouette Score")
ax2.legend(fontsize=8)

# 4c: Davies-Bouldin vs k
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(list(K_RANGE), db_scores, "^-", color="#6a1b9a", linewidth=2, markersize=6)
ax3.axvline(k_opt, color=COLOR_ACC, linestyle="--", label=f"k óptimo = {k_opt}", linewidth=1.5)
ax3.set_title("Davies-Bouldin Score vs k\n(menor = mejor)", fontweight="bold")
ax3.set_xlabel("Número de clusters (k)")
ax3.set_ylabel("Davies-Bouldin Score")
ax3.legend(fontsize=8)

# 4d: PCA scatter por cluster
ax4 = fig.add_subplot(gs[1, :2])
for cl in sorted(df_ml["cluster"].unique()):
    mask = df_ml["cluster"] == cl
    ax4.scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        c=[CLUSTER_PALETTE[cl]], label=f"Cluster {cl}",
        alpha=0.4, s=12, edgecolors="none",
    )
centroides_pca = pca.transform(kmeans_final.cluster_centers_)
ax4.scatter(
    centroides_pca[:, 0], centroides_pca[:, 1],
    c="black", marker="X", s=150, zorder=5, label="Centroides",
)
ax4.set_title(
    f"Proyección PCA 2D de Clusters (k={k_opt})\n"
    f"Varianza explicada: PC1={var_exp[0]:.1%} + PC2={var_exp[1]:.1%}",
    fontweight="bold",
)
ax4.set_xlabel(f"PC1 ({var_exp[0]:.1%})")
ax4.set_ylabel(f"PC2 ({var_exp[1]:.1%})")
ax4.legend(fontsize=8, markerscale=2)

# 4e: Boxplot precio por cluster
ax5 = fig.add_subplot(gs[1, 2])
data_box = [
    df_ml[df_ml["cluster"] == c]["price"].values
    for c in sorted(df_ml["cluster"].unique())
]
bp = ax5.boxplot(data_box, patch_artist=True, medianprops=dict(color="black", linewidth=2))
for patch, color in zip(bp["boxes"], CLUSTER_PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
p95 = df_ml["price"].quantile(0.95)
ax5.set_ylim(0, p95)
ax5.set_title("Distribución de Precio por Cluster\n(sin outliers >p95)", fontweight="bold")
ax5.set_xlabel("Cluster")
ax5.set_ylabel("Precio (MXN/noche)")
ax5.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax5.set_xticklabels([f"C{c}" for c in sorted(df_ml["cluster"].unique())])

# 4f: Perfil de centroides normalizados
ax6 = fig.add_subplot(gs[2, :])
centros_df = pd.DataFrame(
    kmeans_final.cluster_centers_,
    columns=FEATURES,
    index=[f"Cluster {i}" for i in range(k_opt)],
)
cent_norm = (centros_df - centros_df.min()) / (centros_df.max() - centros_df.min() + 1e-9)
feat_labels = [f.replace("_", "\n") for f in FEATURES]
x_pos = np.arange(len(FEATURES))
width = 0.8 / k_opt
for i, (idx, row) in enumerate(cent_norm.iterrows()):
    offset = (i - k_opt / 2 + 0.5) * width
    ax6.bar(
        x_pos + offset, row.values, width,
        label=idx, color=CLUSTER_PALETTE[i], alpha=0.8,
    )
ax6.set_xticks(x_pos)
ax6.set_xticklabels(feat_labels, fontsize=8)
ax6.set_ylabel("Valor normalizado [0-1]")
ax6.set_title("Perfil de Centroides por Cluster (valores normalizados)", fontweight="bold")
ax6.legend(fontsize=8, ncol=k_opt, loc="upper right")
ax6.set_ylim(0, 1.15)

fig.suptitle(
    f"Punto 4 – Segmentación K-Means (k={k_opt}) – Airbnb CDMX",
    fontsize=14, fontweight="bold",
)
save_fig("04_kmeans_clustering")


  PCA – varianza explicada: PC1=0.287, PC2=0.159 (total=0.445)


  [FIG guardada] → v1_visualizations\04_kmeans_clustering.png


## 7. Interpretación del Modelo y Hallazgos


In [66]:
precio_medio_cluster = df_ml.groupby("cluster")["price"].mean().sort_values()
cluster_barato = precio_medio_cluster.index[0]
cluster_caro   = precio_medio_cluster.index[-1]



El modelo K-Means con k=2 logró un Silhouette Score de 0.4182 y Davies- Bouldin de 1.4388. Scores de Silhouette entre 0.1 y 0.3 son habituales en datasets de alojamientos con alta variabilidad de precios, ya que los clusters no son perfectamente separables en el espacio original. El Cluster 0 concentra los alojamientos de menor precio (promedio $1,111 MXN/noche) mientras que el Cluster 1 agrupa propiedades premium (promedio $3,654 MXN/noche).

La proyección PCA captura parte limitada de la varianza total, lo que indica que los clusters no son perfectamente linealmente separables — esperado dado que el precio tiene distribución sesgada y las variables de demanda (number_of_reviews, reviews_per_month) tienen escalas muy distintas. El perfil de centroides revela que 'bedrooms', 'accommodates' y 'price' son las variables que más diferencian los segmentos, mientras que 'minimum_nights' y 'availability_365' contribuyen menos a la separación.

## 8. Limitaciones y Reflexión Crítica

 1. **K-Means asume clusters esféricos** y es sensible a outliers de precio —
    los alojamientos >p95 pueden distorsionar centroides.
 2. La codificación numérica de `room_type` y `neighbourhood` no refleja
    distancias semánticas reales.
 3. Variables cualitativas importantes (descripción, amenities, fotos)
    no están incluidas.
 4. **Alternativas futuras:** DBSCAN (sin asumir forma esférica) o clustering
    jerárquico para explorar estructura anidada por alcaldía.


## 9. Resumen Final


In [67]:


sep("EJECUCIÓN COMPLETADA")
print(f"  Base de datos : {DB_PATH}")
print(f"  Salida figuras: {OUTPUT_DIR}/")
print(f"  k óptimo      : {k_opt}")
print(f"  Registros ML  : {len(df_ml):,}")
print(f"  Silhouette    : {sil_final:.4f}")
print(f"  Davies-Bouldin: {db_final:.4f}")
print(f"  Calinski-Harab: {ch_final:.2f}")
print()
print("  Distribución final por cluster:")
for c in sorted(df_ml["cluster"].unique()):
    n   = (df_ml["cluster"] == c).sum()
    pct = n / len(df_ml) * 100
    p_med = df_ml.loc[df_ml["cluster"] == c, "price"].median()
    print(f"    Cluster {c}: {n:>5,} listings ({pct:.1f}%)  |  precio mediano: ${p_med:,.0f}")


========================= EJECUCIÓN COMPLETADA =========================

  Base de datos : ../data/airbnb.db
  Salida figuras: v1_visualizations/
  k óptimo      : 2
  Registros ML  : 21,018
  Silhouette    : 0.4182
  Davies-Bouldin: 1.4388
  Calinski-Harab: 3683.70

  Distribución final por cluster:
    Cluster 0: 18,667 listings (88.8%)  |  precio mediano: $950
    Cluster 1: 2,351 listings (11.2%)  |  precio mediano: $2,458
